# 1a. The model: problem, gene-level model, projection

Slide figures (16:9, inserted at 100 %), one per concept. Notation as in the code (`clocksync/_core/plane.py`,
`factor.py`), except the gene acrophases, written θ_g here (ψ_g in the code) to keep ψ_k for the compartment noise.

| symbol | meaning |
|---|---|
| i, k, g | donor, compartment, clock gene (14) |
| y_{g,ik} | log fraction log(count / genome-wide depth) of gene g in pseudobulk (i, k) |
| a_g, θ_g | gene amplitude and acrophase of the GTEx waveform (fixed) |
| X_g = a_g (cos θ_g, sin θ_g) | gene row of the clock design |
| β_ik = λ_ik (cos φ_ik, sin φ_ik) | clock state of pseudobulk (i, k): amplitude λ, phase φ |
| δ_kg, s_ik | constant offset of compartment k on gene g; common shift of the pseudobulk |
| L | projector, β = L y |

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch
import clocksync as cs
from clocksync.paired import projector
from clocksync.waveform import HOURS
import paperfig as pf

pf.use_style("slide")
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
wf = cs.gtex()
C = pf.OKABE_ITO
INK, MUTED = "#222222", pf.GREY

## Figure M1: the problem

Every donor is sampled once, at an unrecorded hour. All compartments of a donor share that moment (the host time);
each compartment reads it with its own constant offset and amplitude, and with its own noise. Hands: compartment
clock phase = donor time + compartment offset (+ noise for the T cell). Illustrative values.

In [2]:
COMPS = [("fibroblast", 0.0, C[0]), ("endothelial", 2.0, C[1]), ("hepatocyte", 3.0, C[2]), ("T cell", -3.0, C[3])]
DONORS = [("donor 1", 9.0), ("donor 2", 14.0), ("donor 3", 17.5)]
noise = np.random.default_rng(3).normal(0, 1.8, len(DONORS))  # T cell: noisy clock


def clock(ax, x, y, hour, color, r=5.0):
    ax.add_patch(Circle((x, y), r, fc="white", ec=MUTED, lw=1.0))
    for h in (0, 6, 12, 18):
        a = np.pi / 2 - 2 * np.pi * h / 24
        ax.plot([x + 0.82 * r * np.cos(a), x + r * np.cos(a)], [y + 0.82 * r * np.sin(a), y + r * np.sin(a)], color=MUTED, lw=0.8)
    a = np.pi / 2 - 2 * np.pi * hour / 24
    ax.annotate("", xy=(x + 0.8 * r * np.cos(a), y + 0.8 * r * np.sin(a)), xytext=(x, y),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0, shrinkA=0, shrinkB=0, mutation_scale=10))
    ax.add_patch(Circle((x, y), 0.5, color=color))


lay = pf.Layout.slide("A", "wide", blank="A")
ax = lay.axes["A"]
x0, dx, y0, dy = 62, 34, 76, 22
for j, (name, off, col) in enumerate(COMPS):
    ax.text(x0 + j * dx, y0 + 10, name, ha="center", va="bottom", color=col, fontsize=13)
for i, (donor, hour) in enumerate(DONORS):
    y = y0 - i * dy
    ax.text(4, y + 1.5, donor, ha="left", va="center", fontsize=13, color=INK)
    ax.text(4, y - 3.5, f"sampled at ? h  (true {hour:g} h)", ha="left", va="center", fontsize=10, color=MUTED)
    for j, (name, off, col) in enumerate(COMPS):
        clock(ax, x0 + j * dx, y, hour + off + (noise[i] if name == "T cell" else 0), col)
hl = dict(boxstyle="round,pad=0,rounding_size=3", lw=0, zorder=0)
ax.add_patch(FancyBboxPatch((x0 - 9, y0 - 8), 3 * dx + 18, 16, fc="#eef2f7", **hl))  # donor 1: one moment
xc = x0 + 2 * dx  # hepatocyte column: one compartment
ax.add_patch(FancyBboxPatch((xc - 8, y0 - 2 * dy - 8), 16, 2 * dy + 16 + 10, fc="#e9f4ef", **hl))
xr = x0 + 3 * dx + 14
ax.text(xr, y0 + 2.5, "one donor, one sampling moment", va="center", fontsize=12, color=INK)
ax.text(xr, y0 - 3, r"$\rightarrow$ shared host state $f_i$ (time unknown)", va="center", fontsize=12, color=INK)
ax.text(xr, y0 - 2 * dy + 2.5, "one compartment, every donor", va="center", fontsize=12, color=C[2])
ax.text(xr, y0 - 2 * dy - 3, r"$\rightarrow$ constant offset and amplitude $z_k$, own noise $\psi_k$", va="center", fontsize=12, color=C[2])
ax.plot([xc + 8, xr - 2], [y0 - 2 * dy, y0 - 2 * dy], color=C[2], lw=0.8, ls=":")
ax.set(xlim=(0, 320), ylim=(18, 96))
ax.set_aspect("equal", adjustable="box"); ax.set_anchor("C")
fig = lay.finalize()
pf.save(fig, FIG / "M1_problem")
fig

<Figure size 3779.53x1299.21 with 1 Axes>

## Figure M2: gene-level model

The log expression of a clock gene is a gene baseline, a constant compartment offset, a common shift of the
pseudobulk and a clock term. The clock term is linear in the 2-D clock state β_ik through the fixed GTEx gene
loadings X_g (right). Assumption: the waveform shape (relative amplitudes a_g and acrophases θ_g) is the same in every
compartment and donor; only the amplitude λ and phase φ of the clock state change.

In [3]:
lay = pf.Layout.slide("EEP", "wide", blank="E", polar="P")
ax = lay.axes["E"]
pf.equation(ax, (0.5, 0.8), [r"\log\,\mathrm{E}[y_{g,ik}]", "=", (r"m_g", "gene baseline", C[0]), "+",
            (r"\delta_{kg}", "compartment offset", C[1]), "+", (r"s_{ik}", "sample shift\n(depth, RNA)", C[2]), "+",
            (r"\lambda_{ik}\, a_g \cos(\phi_{ik} - \theta_g)", "clock term", C[3])])
pf.equation(ax, (0.5, 0.42), [(r"\lambda_{ik}\, a_g \cos(\phi_{ik} - \theta_g)", None, C[3]), "=",
            (r"X_g\, \beta_{ik}", "linear in the clock state", C[3])])
pf.equation(ax, (0.5, 0.1), [(r"X_g = a_g\,(\cos\theta_g,\ \sin\theta_g)", "fixed gene loadings (GTEx)", MUTED),
            r"\qquad", (r"\beta_{ik} = \lambda_{ik}\,(\cos\phi_{ik},\ \sin\phi_{ik})^{T}", "clock state of the sample", C[3])])
pf.polar_phase(lay.axes["P"], np.asarray(wf.psi) * HOURS, wf.amp, arrows=True, labels=list(wf.genes),
               colors=[MUTED] * len(wf.genes), title=r"Fixed waveform: $a_g$, $\theta_g$ (h)")
fig = lay.finalize()
pf.save(fig, FIG / "M2_gene_model")
fig

<Figure size 3779.53x1299.21 with 2 Axes>

## Figure M3: the projection

β_ik = L y_ik, with one projector per (study, compartment). P centres over genes; W holds inverse-variance gene weights
from the compartment's median counts (Poisson + τ², τ² = 0.05). Three properties follow from the construction
(`plane.projector`: L X = I and L 1 = 0).

In [4]:
lay = pf.Layout.slide("E", "half", blank="E")
ax = lay.axes["E"]
pf.equation(ax, (0.5, 0.9), [(r"\beta_{ik}", "clock state", C[3]), "=", (r"L", "projector", INK),
            (r"y_{ik}", "log fractions", C[0])])
pf.equation(ax, (0.5, 0.66), [r"L", "=", r"(\tilde{X}^{T} W \tilde{X})^{-1}\, \tilde{X}^{T}", (r"W", "gene weights\n(one per compartment)", C[2]),
            (r"P", "centring\nover genes", C[1]), ",", r"\tilde{X} = P X"])
rows = [(r"L\,\mathbf{1} = 0", r"\Rightarrow", "the sample shift $s_{ik}$ cancels", C[2]),
        (r"L\,X = I", r"\Rightarrow", r"the clock term gives $\beta_{ik} = \lambda_{ik}(\cos\phi_{ik}, \sin\phi_{ik})$", C[3]),
        (r"L\,\delta_k = \mu_k", r"\Rightarrow", "constant offsets shift $\\beta$ by a constant", C[1])]
for j, (a, b, c, col) in enumerate(rows):
    pf.equation(ax, (0.06, 0.4 - 0.11 * j), [(a, None, col), b, dict(tex=c, math=False)], ha="left")
pf.equation(ax, (0.5, 0.04), [(r"V_{ik}", "Poisson covariance of $\\beta_{ik}$", MUTED), "=",
            r"L\, \mathrm{diag}(1/\mathrm{count}_{g,ik})\, L^{T}"])
fig = lay.finalize()
pf.save(fig, FIG / "M3_projection_formula")
fig

<Figure size 2244.09x1653.54 with 1 Axes>

## Figure M4: what the projection does

Noise-free log fractions for 24 samples over the day (one per hour), projected with the unweighted projector of the
GTEx waveform (`clocksync.paired.projector`). **A** the clock term alone traces a circle of radius λ (a gene-independent baseline cancels; a per-gene baseline m_g acts like the offset of C, common to all compartments).
**B** a random common shift per sample (sd 0.5) leaves every β unchanged. **C** a random per-gene compartment offset
(sd 1.5) translates the whole circle by the constant μ_k = L δ_k, the same for every sample of that compartment:
the factor model's intercept absorbs it. Only relative clock states remain.

In [5]:
genes = list(wf.genes)
L = projector(pd.DataFrame(np.ones((1, len(genes))), columns=genes), wf, counts=False)
rng = np.random.default_rng(0)
hours = np.arange(24.0)
phi = hours / HOURS
clock_term = np.column_stack([np.cos(phi), np.sin(phi)]) @ wf.X.T  # lambda = 1
base = np.full(len(genes), -9.0)  # gene-independent part (a per-gene baseline m_g acts like an offset, panel C)
shift = rng.normal(0, 0.5, (len(hours), 1))
delta = rng.normal(0, 1.5, len(genes))
proj = lambda y: y @ L.T
B0, B1, B2 = proj(base + clock_term), proj(base + clock_term + shift), proj(base + clock_term + delta)
mu = L @ delta
print("max |beta change| from the sample shift:", np.abs(B1 - B0).max().round(12), " mu_k =", mu.round(3))

lay = pf.Layout.slide("ABC", "wide", wgap=10, aspect={"A": 1, "B": 1, "C": 1}, anchor="C")
ax = lay.axes
cmap = plt.get_cmap("twilight_shifted")
lim = (-2.7, 2.7)
for key, B, title in (("A", B0, "clock term"), ("B", B1, "+ sample shift $s_{ik}$"), ("C", B2, r"+ compartment offset $\delta_{kg}$")):
    a = ax[key]
    a.axhline(0, color="#dddddd", lw=0.8, zorder=0); a.axvline(0, color="#dddddd", lw=0.8, zorder=0)
    if key != "A":
        a.scatter(B0[:, 0], B0[:, 1], s=40, facecolors="none", edgecolors="#bbbbbb", lw=0.8)
    sc = a.scatter(B[:, 0], B[:, 1], c=hours, cmap=cmap, vmin=0, vmax=24, s=22, linewidths=0, zorder=3)
    a.set(xlim=lim, ylim=lim, aspect="equal", title=title, xlabel=r"$\beta_1$", ylabel=r"$\beta_2$" if key == "A" else None)
ax["B"].text(0.03, 0.97, r"$L\,\mathbf{1} = 0$: unchanged", transform=ax["B"].transAxes, fontsize=11, color=C[2], va="top")
ax["C"].annotate("", xy=mu, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=C[1], lw=1.6, mutation_scale=10))
ax["C"].text(0.03, 0.97, r"shift $\mu_k = L\,\delta_k$", transform=ax["C"].transAxes, fontsize=11, color=C[1], va="top")
pf.colorbar(ax["C"], sc, "sample hour (h)")
fig = lay.finalize()
pf.save(fig, FIG / "M4_projection_effects")
fig

max |beta change| from the sample shift: 0.0  mu_k = [1.276 0.497]


<Figure size 3779.53x1299.21 with 4 Axes>